In [2]:
import json
import sys
from rich import print as rprint
from rich import inspect
from pathlib import Path
import re

nb_dir = Path.cwd()

project_root = nb_dir.parent.parent

sys.path.insert(0, str(project_root))


from scripts.text_matching import normalise_text

In [3]:
people_file = Path(project_root / "data/from db/people.json")
variants_file = Path(project_root / "data/from db/people_variants.json")
unmatched_file = Path(project_root / "data/people/unmatched.json")

prepped_books_folder = Path(project_root / "data/prepped")

# test_book_file = Path(project_root / "data/prepped/antike.json")

In [4]:
with open(unmatched_file, "r") as f:
   unmatched = json.load(f)

with open(variants_file, "r") as f:
   variants = json.load(f)

with open(people_file, "r") as f:
   people = json.load(f)

# with open(test_book_file, "r") as f:
#    entries = json.load(f)


In [5]:
books_dict = {}
for file in prepped_books_folder.iterdir():

    if not file.exists():
       raise FileNotFoundError(f"{file} doesn't exist!")

    with open(file, "r") as f:
       entries = json.load(f)

    for composite_id, book_data in entries.items():
        original_entry = book_data[0]["admin_data"]["original_entry"]
        notes = book_data[0]["admin_data"].get("verification_notes", "")

        books_dict[composite_id] = {
            "original_entry": original_entry,
            "verification_notes": notes
        }

# with open("books_dict_notes.json", "w") as f:
#     json.dump(books_dict, f, ensure_ascii=False, indent=2)
# rprint(dict(list(books_dict.items())[:4]))

In [6]:
multiples_dict = {}
orgs_dict = {}
unmatched_others_dict = {}

SEPARATORS = re.compile(r"\s*;\s*|\s+/\s+|\s+und\s+")
ETC_SUFFIX = re.compile(r"\s+u\.a\.\s*$")

org_keywords = ["stiftung", "archiv", "gesellschaft", "forum", "gemeinde", "sammlung", "institut", "museum", "kultur", "kuratorium", "verlag", "ministerium", "akademie", "trust", "verein", "organisation", "vereinigung", "universität", "bibliothek", "stadt", "staatlich", "werkstatt", ]

single_name_count = 0

for name, entries in unmatched.items():
    if any(keyword in name for keyword in org_keywords):
        orgs_dict[name] = entries
    elif SEPARATORS.search(name):
        multiples_dict[name] = entries
    elif ETC_SUFFIX.search(name):
        multiples_dict[name] = entries
    else:
        if " " not in name:
            single_name_count += 1
        unmatched_others_dict[name] = entries

rprint(f"total unmatched: {len(unmatched)}")
rprint(f"total multiples: {len(multiples_dict)}")
rprint(f"total orgs: {len(orgs_dict)}")
rprint(f"total others: {len(unmatched_others_dict)}")
rprint(f"single names counted: {single_name_count}")

rprint(f"is the math mathing? Sum of all: {len(unmatched_others_dict) + len(orgs_dict) + len(multiples_dict)}")

# with open("multiples_manual_fixes.json", "w") as f:
#     json.dump(multiples_dict, f, ensure_ascii=False, indent=2)
# with open("unmatched_manual_edits.json", "w") as f:
#     json.dump(unmatched_others_dict, f, ensure_ascii=False, indent=2)


total unmatched: 1575

total multiples: 34

total orgs: 18

total others: 1523

single names counted: 106

is the math mathing? Sum of all: 1575

In [7]:
# for names, data in multiples_dict.items():
with open("multiples_manual_fixes.json", "r") as f:
   multiples_dict = json.load(f)

with open("unmatched_manual_edits.json", "r") as f:
   unmatched_others_dict = json.load(f)

# rprint(f"unmatched before merge: {len(unmatched_others_dict)}")


for data in multiples_dict.values():

    name_string = data[0]["display_name"]

    cleaned = ETC_SUFFIX.sub("", name_string).strip()
    parts = SEPARATORS.split(cleaned)

    shared_last = None
    for p in parts:
        if "," in p:
            shared_last = p.split(",")[0].strip()
            # rprint(f"shared: {shared_last}")
            break

    complete_names = []
    for p in parts:

        if "," not in p and shared_last:
            complete_names.append(f"{shared_last}, {p.strip()}")
        else:
            complete_names.append(p.strip())
    # rprint(f"parts after loop: {parts}")

    for i, name in enumerate(complete_names):

        # rprint(f"name in complete: {i, name}")
        sorting_new = i + 1
        name_norm = normalise_text(name)

        if name_norm not in unmatched_others_dict:
            unmatched_others_dict[name_norm] = [{
                **data[0],
                "display_name": name,
                "sort_order": sorting_new
            }]

        else:
            unmatched_others_dict[name_norm].append({
                **data[0],
                "display_name": name,
                "sort_order": sorting_new
            })


# rprint(f"unmatched after merge: {len(unmatched_others_dict)}")

# with open("unmatched_merged.json", "w") as f:
#     json.dump(unmatched_others_dict, f, ensure_ascii=False, indent=2)


In [8]:
with open("unmatched_merged.json", "r") as f:
   unmatched_others_dict = json.load(f)


for name, entries in orgs_dict.items():
    is_organisation = True

    if name not in unmatched_others_dict:
        unmatched_others_dict[name] = [{
            **data[0],
            "is_organisation": is_organisation
        }]

else:
    unmatched_others_dict[name].append({
        **data[0],
        "is_organisation": is_organisation
    })

rprint(f"unmatched with orgs: {len(unmatched_others_dict)}")

# with open("unmatched_merged_orgs.json", "w") as f:
#     json.dump(unmatched_others_dict, f, ensure_ascii=False, indent=2)

unmatched with orgs: 1605

In [9]:
import csv


with open("unmatched_merged_orgs.json", "r") as f:
   unmatched_dict = json.load(f)
# rprint(dict(list(unmatched_dict.items())[:2]))

unmatched_flat = []
last = first = single = None

for name, data_list in unmatched_dict.items():
    for entry in data_list:
        composite_id = entry["composite_id"]
        display_name = entry["display_name"]
        display_norm = normalise_text(display_name)
        is_organisation = entry.get("is_organisation", False)

        if not is_organisation:
            if ", " in display_name:
                last, first = display_name.split(", ", 1)
            else:
                split = display_name.rsplit(" ", 1)
                if len(split) == 2:
                    first, last = split
                else:
                    single = display_name
        else:
            single = display_name

        original_entry = books_dict[composite_id]["original_entry"]
        notes = books_dict[composite_id]["verification_notes"]


        unmatched_flat.append({
            "composite_id": composite_id,
            **entry,
            "is_organisation": is_organisation,
            "display_norm": display_norm,
            "last": last,
            "first": first,
            "single": single,
            "original_entry": original_entry,
            "verification_notes": notes
        })

# rprint(unmatched_flat[:4])

# with open("unmatched_flat.json", "w") as f:
#     json.dump(unmatched_flat, f, ensure_ascii=False, indent=2)

# fieldnames = list(matched_flat[0].keys())
# rprint(fieldnames)

with open("unmatched_flat.csv", mode="w") as csv_file:
    fieldnames = list(unmatched_flat[0].keys())
    writer = csv.DictWriter(csv_file, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(unmatched_flat)